# Day 19: Python gotchas, rotated search, attention

*Week 4: ML from scratch*

**Time budget:** about 60–90 minutes. **A** is Python fluency (~15 min), **B** is an interview-style algorithm (~25 min), **C** is NumPy / PyTorch / ML (~30 min).

**For each problem:**
1. Plan on paper or in comments before you type any code.
2. Work on it for 20 minutes before asking for a hint.
3. Run the test cell under it. Don't edit the tests.
4. Answer the **Say out loud** prompts, out loud. Interviews grade the explanation as much as the code.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## A. Predict the output

`Python`

Each snippet below sets a variable `result`. In the code cell, fill in your **prediction** for each one, **without running anything first**. Then run the test cell: it executes each snippet and explains the ones you got wrong.

Interviewers use exactly these to check that you understand Python's object model.

```
1.  a = [1, 2, 3]; b = a; b.append(4); result = a
2.  grid = [[0] * 3] * 3; grid[0][0] = 1; result = grid
3.  fs = [lambda x: x * i for i in range(3)]; result = [f(10) for f in fs]
4.  result = ((-7) // 2, (-7) % 2)
5.  result = (type((5)).__name__, type((5,)).__name__)
6.  a = [1, 2]; b = a; a += [3]; result = b
7.  a = [1, 2]; b = a; a = a + [3]; result = b
8.  result = 0.1 + 0.2 == 0.3
9.  d = {"b": 1, "a": 2}; d["c"] = 3; result = list(d)
10. def f(x, acc=[]): acc.append(x); return acc
    f(1); result = f(2)
11. x = np.array([1, 2, 3]); y = x[1:]; y[0] = 99; result = x.tolist()
12. t = torch.ones(3); u = t.numpy(); u[0] = 5; result = t.tolist()
13. x = np.arange(6).reshape(2, 3); result = (x + np.array([10, 20, 30])).tolist()
14. result = sorted(["b", "A", "c"])
```

In [ ]:
predictions = {
    1: None,
    2: None,
    3: None,
    4: None,
    5: None,
    6: None,
    7: None,
    8: None,
    9: None,
    10: None,
    11: None,
    12: None,
    13: None,
    14: None,
}

In [ ]:
# Tests: run after your solution. Don't edit.
import torch
SNIPPETS = {
    1: ("a = [1, 2, 3]; b = a; b.append(4); result = a", "b = a copies the reference, not the list."),
    2: ("grid = [[0] * 3] * 3; grid[0][0] = 1; result = grid", "* 3 on the outer list repeats the SAME inner list 3 times."),
    3: ("fs = [lambda x: x * i for i in range(3)]; result = [f(10) for f in fs]", "Late binding: each lambda looks up i when called, after the loop ended (i == 2)."),
    4: ("result = ((-7) // 2, (-7) % 2)", "// floors toward -infinity, and % takes the sign of the divisor."),
    5: ("result = (type((5)).__name__, type((5,)).__name__)", "Parentheses alone don't make a tuple; the comma does."),
    6: ("a = [1, 2]; b = a; a += [3]; result = b", "+= on a list mutates it in place, so b sees the change."),
    7: ("a = [1, 2]; b = a; a = a + [3]; result = b", "a + [3] builds a NEW list; b still points at the old one."),
    8: ("result = 0.1 + 0.2 == 0.3", "Floating point: 0.1 + 0.2 is 0.30000000000000004. Use math.isclose / np.isclose."),
    9: ('d = {"b": 1, "a": 2}; d["c"] = 3; result = list(d)', "Dicts keep insertion order (Python 3.7+)."),
    10: ("def f(x, acc=[]):\n    acc.append(x)\n    return acc\nf(1); result = f(2)", "The default list is created once, when the function is defined (Day 3)."),
    11: ("x = np.array([1, 2, 3]); y = x[1:]; y[0] = 99; result = x.tolist()", "NumPy slices are VIEWS that share memory (unlike list slices)."),
    12: ("t = torch.ones(3); u = t.numpy(); u[0] = 5; result = t.tolist()", ".numpy() on a CPU tensor shares memory with the tensor."),
    13: ("x = np.arange(6).reshape(2, 3); result = (x + np.array([10, 20, 30])).tolist()", "Broadcasting: the (3,) array is added to each row."),
    14: ('result = sorted(["b", "A", "c"])', "Uppercase letters sort before lowercase (by character code)."),
}
wrong = 0
for k, (code, why) in SNIPPETS.items():
    ns = {"np": np, "torch": torch}
    exec(code, ns)
    if predictions.get(k) != ns["result"]:
        wrong += 1
        print(f"❌ {k}: you said {predictions.get(k)!r}, actual {ns['result']!r}\n   why: {why}")
print(f"\n{len(SNIPPETS) - wrong}/{len(SNIPPETS)} correct")
if wrong == 0:
    print("✅ A passed")

## B. Search in a rotated sorted array

`Algorithms`

A sorted list of **distinct** numbers was rotated at an unknown point, e.g. `[0, 1, 2, 4, 5, 6, 7]` became `[4, 5, 6, 7, 0, 1, 2]`. Write `search_rotated(nums, target)`: return the index of `target`, or -1. It must be **O(log n)**.

The key observation: when you split at `mid`, at least **one** half is properly sorted. Work out which one, check whether `target` lies inside its range, and continue in the correct half.

In [ ]:
def search_rotated(nums, target):
    # your code here
    pass

In [ ]:
# Tests: run after your solution. Don't edit.
assert search_rotated([4, 5, 6, 7, 0, 1, 2], 0) == 4
assert search_rotated([4, 5, 6, 7, 0, 1, 2], 3) == -1
assert search_rotated([1], 0) == -1 and search_rotated([1], 1) == 0
base = list(range(0, 40, 2))
for r in range(len(base)):
    nums = base[r:] + base[:r]
    for t in range(-1, 41):
        expected = nums.index(t) if t in nums else -1
        assert search_rotated(nums, t) == expected, (nums, t)
print("✅ B passed")

## C. Scaled dot-product attention

`ML from scratch`

1. `attention(Q, K, V, mask=None)`: Q is (B, T, d), K is (B, S, d), V is (B, S, dv). Return (B, T, dv) = softmax(Q Kᵀ / √d) V, where the softmax is over the last axis (over keys). `mask` is a boolean (T, S) or (B, T, S) tensor where **`True` means "allowed to attend"**. Set the scores at disallowed positions to `-inf` **before** the softmax.
2. `causal_mask(T)`: a (T, T) boolean mask where position i may only attend to positions ≤ i.

You may use `torch.softmax`, but not `F.scaled_dot_product_attention` (the test uses it to check you).

**Look up:** `masked_fill`, `torch.tril`, `torch.ones(..., dtype=torch.bool)`.

**Say out loud:** why divide by √d? What's the cost in T? **Stretch:** wrap it in a multi-head `nn.Module`.

In [ ]:
def attention(Q, K, V, mask=None):
    # your code here
    pass


def causal_mask(T):
    # your code here
    pass

In [ ]:
# Tests: run after your solution. Don't edit.
torch.manual_seed(0)
Q, K, V = torch.randn(2, 5, 8), torch.randn(2, 7, 8), torch.randn(2, 7, 4)
out = attention(Q, K, V)
assert out.shape == (2, 5, 4)
assert torch.allclose(out, F.scaled_dot_product_attention(Q, K, V), atol=1e-5)
m = causal_mask(4)
assert m.dtype == torch.bool and m.shape == (4, 4)
assert torch.equal(m, torch.tensor([[1, 0, 0, 0], [1, 1, 0, 0], [1, 1, 1, 0], [1, 1, 1, 1]], dtype=torch.bool))
Q, K, V = torch.randn(3, 6, 8), torch.randn(3, 6, 8), torch.randn(3, 6, 8)
out = attention(Q, K, V, mask=causal_mask(6))
assert torch.allclose(out, F.scaled_dot_product_attention(Q, K, V, attn_mask=causal_mask(6)), atol=1e-5)
assert torch.allclose(out[:, 0], V[:, 0], atol=1e-6), "the first position can only see itself"
print("✅ C passed")

## Wrap-up (5 minutes)

- For each problem: what's its time and space complexity? Which edge cases did you handle?
- Anything that took more than 20 minutes, or needed a hint: add it to the **Redo list** in `README.md` and solve it again from scratch in 3 days, without looking at today's code.
- Tick off today in `README.md`.